In [1]:
!pip install ipynb
!pip install matplotlib seaborn pandas numpy
!pip install xgboost

In [2]:
from pathlib import Path

PROJECT_DIRECTORY = Path.cwd().parent

RAW_DATA = PROJECT_DIRECTORY / "data" / "raw" / "healthcare-dataset-stroke-data.csv"

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [4]:
PROJECT_DIRECTORY = Path.cwd().parent
RAW_DATA = (
    PROJECT_DIRECTORY / "data" / "raw" / "healthcare-dataset-stroke-data.csv"
)


class DataCollector:

    def __init__(self):
        self.raw_data = None
        self.X = None
        self.y = None

    def _import_data(self):
        self.raw_data = pd.read_csv(RAW_DATA)
        self.X = self.raw_data.drop(columns=["stroke"])
        self.y = self.raw_data["stroke"]

    def _check_data(self, expected_row_count, expected_col_names):
        assert (
            self.raw_data.shape[0] == expected_row_count
        ), "Row count mismatch, verify row count"
        assert self.raw_data.shape[1] == len(
            expected_col_names
        ), "Column count mismatch, verify column count"
        assert set(expected_col_names).issubset(
            self.raw_data.columns
        ), "Column names mismatch, verify expected columns"

    def split_data(
        self, expected_row_count, expected_col_names, stratify_tol=0.1
    ):
        self._import_data()
        self._check_data(expected_row_count, expected_col_names)

        # Data Split: 70/15/15 raw data partitions
        X_train, X_dev, y_train, y_dev = train_test_split(
            self.X,
            self.y,
            test_size=0.30,
            stratify=self.y,
            random_state=42,
        )

        X_val, X_test, y_val, y_test = train_test_split(
            X_dev,
            y_dev,
            test_size=0.50,
            stratify=y_dev,
            random_state=42,
        )

        X_splits = {"X_train": X_train, "X_val": X_val, "X_test": X_test}
        y_splits = {"y_train": y_train, "y_val": y_val, "y_test": y_test}

        # Data Split Checks
        assert (
            len(self.X)
            == len(self.y)
            == sum(len(split) for split in X_splits.values())
            == sum(len(split) for split in y_splits.values())
        ), "Partition row count MISMATCH with original dataset"
        print("Partition row count match with original dataset.")

        # Check class stratification proportionality
        for y_name, y_split in y_splits.items():
            assert np.allclose(
                y_split.value_counts(normalize=True),
                self.y.value_counts(normalize=True),
                atol=stratify_tol,
            ), f"{y_name} target preservation fail"
        print("Class proportions reasonably preserved.")

        # Check overlap between splits
        combined_splits = [("X", X_splits), ("y", y_splits)]
        for _, splits in combined_splits:
            split_items = list(splits.items())
            for i1 in range(len(split_items)):
                for i2 in range(i1 + 1, len(split_items)):
                    split_name1, split_data1 = split_items[i1]
                    split_name2, split_data2 = split_items[i2]

                    assert set(split_data1.index).isdisjoint(
                        set(split_data2.index)
                    ), f"Overlap detected between {split_name1} and {split_name2}"

        return X_splits, y_splits